In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import gpt

In [ ]:
ROOT = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\30papers'
#ROOT = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Natalie\pdfs'

In [ ]:
pdf_paths = []
for current_dir, subdirs, files in os.walk(ROOT):
    for file in files:
        path = os.path.join(current_dir, file)
        if path.endswith('.pdf'):
            pdf_paths.append(path)

pdf_paths

In [ ]:
client = gpt._CLIENT

In [ ]:
file = client.files.create(
    file=open(pdf_paths[0], "rb"),
    purpose="user_data"
)

In [ ]:
files_with_paths = []
for path in tqdm(pdf_paths):
    files_with_paths.append({'path': path,
        'file_object': client.files.create(
        file=open(path, "rb"),
        purpose="user_data")})


In [ ]:
len(files_with_paths)

In [ ]:
system_prompt = f'''
You are a scientist. You will be given several PDF files of scientific articles. You need to decide whether to include each one of them in a study.

You will be given criteria to decide whether to include or to exclude.

For each article, please print the article's title, then your reasoning, then the decision whether to include or exclude, in the following format:

```
TITLE
<article's title>

REASONING
<reasoning why to include or to exclude>

DECISION
<one of INCLUDE or EXCLUDE>

TITLE
<next article's title>
```


etc. 
You must do this for **each** input.

Write out the decisions for all inputs **in the same order as they are given**.
'''

In [ ]:
def run_model_on_batch(batch):
    content = [
                {
                    "type": "input_text",
                    "text": gpt.PAUL_CRITERIA,
                },]
    for item in batch:
        content.append({ "type": "input_file", "file_id": item['file_object'].id,})
    response = client.responses.create(
        model=gpt.Models.MINI,
        input=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": content,
            }
        ]
    )
    return response.output_text

In [ ]:
OUTPUTS = None
TO_PROCESS = None

In [ ]:
def run_on_missing_items(items, batch_size):
    global OUTPUTS, TO_PROCESS
    to_process = [item for item in items if 'title' not in item]
    TO_PROCESS = to_process
    print(f'Running on {len(to_process)} articles')

    outputs = []
    batches = list(batched(to_process, batch_size))
    try:
        for batch in tqdm(batches):
            outputs.append(run_model_on_batch(batch))
    except BaseException as e:
        print(f'Error: {type(e)}, {e}')
        print('Current files:', *[item['path'] for item in batch], sep='\n')

    OUTPUTS = outputs
    missing_items = 0
    updated_items = 0

    for i, (output, batch) in enumerate(zip(outputs, batches)):
        try:
            res = parse_pdf_response(output)
        except Exception as e:
            print(f'Error in batch {i}: {e}')
            missing_items += len(batch)
            continue
        if len(res) != len(batch):
            print(f'Batch {i}: Expected {len(batch)} results but got {len(res)}.')
            missing_items += len(batch)
            continue
        for item, result in zip(batch, res):
            item.update(result)
        updated_items += len(batch)

    print(f'Updated {updated_items} items, missing {missing_items} items.')


In [ ]:
DECISION_DICT = {'INCLUDE': True, 'EXCLUDE': False}

def parse_pdf_response(response: str):
    items = []
    current_type = None

    for line in response.splitlines():
        line = line.strip()
        if not line or line in {'```', '---'}:
            continue
        if line in {'TITLE', 'REASONING', 'DECISION'}:
            current_type = line
            if line == 'TITLE':
                items.append({'title': '', 'reasoning': '', 'decision': ''})
            continue

        if not current_type:
            raise ValueError(f'This line has no key to assign to: "{line}"')
        key = current_type.lower()
        _add_info(items[-1], key, line)

        if current_type == 'DECISION':
            current_type = ''

    for i, item in enumerate(items):
        if not item['title'] or not item['reasoning'] or not item['decision']:
            raise ValueError(f'Bad item {i}: {item}')
        decision = item['decision']
        if decision not in DECISION_DICT:
            raise ValueError(f'Invalid decision in item {i}: {decision}')
        item['decision'] = DECISION_DICT[decision]

    titles = [item['title'] for item in items]
    if len(set(titles)) < len(titles):
        raise ValueError(f'Titles are not unique: {titles}')

    return items

def _add_info(item, key, line):
    current = item[key]
    if current:
        item[key] = '\n'.join([current, line])
    else:
        item[key] = line

In [ ]:
run_on_missing_items(files_with_paths, 5)

In [ ]:
run_on_missing_items(files_with_paths, 3)

In [ ]:
run_on_missing_items(files_with_paths, 1)

doi_10.2340_00015555-3117_4a13b853d0.pdf was not a PDF

In [ ]:
df = pd.DataFrame(files_with_paths)
del df['file_object']
df.rename(columns={'reasoning': 'full_article_reasoning', 'decision': 'full_article_decision'}, inplace=True)
df = df[~df.full_article_decision.isna()]
df.head()

In [ ]:
df.to_excel(fr'{ROOT}/../results.xlsx', index=False)

In [ ]:
df.full_article_decision.sum()

In [ ]:
len(df), df.title.nunique()

In [ ]:
df.drop_duplicates('title', inplace=True)

In [ ]:
df['title_extract'] = df.title.apply(extract_letters)
len(df), df.title_extract.nunique()

In [ ]:
full_article_df = df

In [ ]:
#old_df = pd.read_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\deduplicate_with_eligibility.xlsx")
old_df = pd.read_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\Natalie\tables\ris_with_decision.xlsx")
old_df.head()

In [ ]:
old_df['title_extract'] = old_df.title.apply(extract_letters)

In [ ]:
#full_article_df = pd.read_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\combined_paul100_pdf_analysis.xlsx")

In [ ]:
combined_df = pd.merge(old_df, full_article_df, on='title_extract', how='outer')
len(combined_df)

In [ ]:
combined_df[['title_x', 'title_y']]

In [ ]:
combined_df.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Natalie\tables\combined_deduplicated_pdf_analysis.xlsx', index=False)